# 🔥 Physics-Informed Neural Networks (PINN) for 1D Heat Equation

> **Problem**: Solve the 1D heat conduction equation using a neural network  
> **Method**: Physics-Informed Neural Networks (PINN) — embed the PDE directly into the loss function  
> **Why PINN?**: Traditional solvers (FDM, FEM) need mesh grids; PINN is **mesh-free** and can handle complex geometries

---

## 1. The Heat Equation

The 1D heat equation describes how temperature $u(x,t)$ diffuses over space $x$ and time $t$:

$$
\frac{\partial u}{\partial t} = \alpha \frac{\partial^2 u}{\partial x^2}
$$

where $\alpha$ is the thermal diffusivity.

### Problem Setup

| Component | Definition |
|-----------|-----------|
| Domain | $x \in [0, 1], \; t \in [0, 1]$ |
| Thermal diffusivity | $\alpha = 0.1$ |
| Initial condition | $u(x, 0) = \sin(\pi x)$ |
| Boundary conditions | $u(0, t) = u(1, t) = 0$ |

### Analytical Solution

For these IC/BC, the analytical solution is:

$$
u(x, t) = \sin(\pi x) \cdot e^{-\alpha \pi^2 t}
$$

We will use this to validate the PINN prediction.

In [ ]:
# ========================
# 0. Imports
# ========================
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
from IPython.display import HTML
import torch
import torch.nn as nn

plt.rcParams['font.size'] = 12
plt.rcParams['figure.dpi'] = 120
plt.rcParams['axes.unicode_minus'] = False

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')
print(f'PyTorch version: {torch.__version__}')

---
## 2. Analytical Solution Visualization

Before building the PINN, let's visualize the ground truth.

In [ ]:
# ===============================================
# 2. Analytical Solution
# ===============================================

alpha = 0.1

def analytical_solution(x, t):
    return np.sin(np.pi * x) * np.exp(-alpha * np.pi**2 * t)

# Create mesh grid
nx, nt = 200, 100
x_vals = np.linspace(0, 1, nx)
t_vals = np.linspace(0, 1, nt)
X_grid, T_grid = np.meshgrid(x_vals, t_vals, indexing='ij')
U_exact = analytical_solution(X_grid, T_grid)

print(f'U_exact shape: {U_exact.shape}')
print(f'Temp range: [{U_exact.min():.4f}, {U_exact.max():.4f}]')

In [ ]:
# ===============================================
# 2.1 Analytical Solution: 3D Surface
# ===============================================

fig = plt.figure(figsize=(18, 5))

# 3D Surface
ax1 = fig.add_subplot(131, projection='3d')
surf = ax1.plot_surface(X_grid, T_grid, U_exact, cmap='hot', edgecolor='none', alpha=0.9)
ax1.set_xlabel('x')
ax1.set_ylabel('t')
ax1.set_zlabel('u(x,t)')
ax1.set_title('Analytical Solution u(x,t)', fontweight='bold')
fig.colorbar(surf, ax=ax1, shrink=0.6, label='Temperature')

# Heatmap (contourf)
ax2 = fig.add_subplot(132)
cf = ax2.contourf(X_grid, T_grid, U_exact, levels=50, cmap='hot')
ax2.set_xlabel('x (space)')
ax2.set_ylabel('t (time)')
ax2.set_title('Heatmap: u(x,t)', fontweight='bold')
fig.colorbar(cf, ax=ax2, shrink=0.8, label='Temperature')

# Snapshots at different times
ax3 = fig.add_subplot(133)
snap_times = [0, 0.1, 0.3, 0.5, 1.0]
colors = plt.cm.hot(np.linspace(0.3, 1.0, len(snap_times)))
for t_snap, color in zip(snap_times, colors):
    u_snap = analytical_solution(x_vals, t_snap)
    ax3.plot(x_vals, u_snap, color=color, lw=2, label=f't={t_snap:.1f}')
ax3.set_xlabel('x (space)')
ax3.set_ylabel('u(x,t)')
ax3.set_title('Temperature Profiles at Different Times', fontweight='bold')
ax3.legend()
ax3.set_ylim(-0.05, 1.05)
ax3.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()
print("Temperature starts as sin(πx) and decays exponentially over time.")

---
## 3. Physics-Informed Neural Network (PINN)

### Core Idea

Instead of training on pre-computed data, PINN enforces the **physics** directly:

$$\text{Loss} = \underbrace{\text{MSE}_{\text{PDE}}}_{\text{PDE residual}} + \underbrace{\text{MSE}_{\text{IC}}}_{\text{Initial condition}} + \underbrace{\text{MSE}_{\text{BC}}}_{\text{Boundary conditions}}$$

### PDE Residual

For a prediction $\hat{u}(x,t)$, the residual is:

$$r(x,t) = \frac{\partial \hat{u}}{\partial t} - \alpha \frac{\partial^2 \hat{u}}{\partial x^2}$$

If $\hat{u}$ satisfies the PDE, $r(x,t) \to 0$.

### Architecture

$$
(x, t) \xrightarrow{\text{FC}} \sigma(W_1 x + b_1) \xrightarrow{\text{FC}} \sigma(W_2 x + b_2) \xrightarrow{\text{FC}} \cdots \xrightarrow{\text{FC}} \hat{u}
$$

where $\sigma$ is the activation function (we use $\tanh$).

### Derivatives via Automatic Differentiation

$$\frac{\partial \hat{u}}{\partial t}, \; \frac{\partial \hat{u}}{\partial x}, \; \frac{\partial^2 \hat{u}}{\partial x^2} \quad \text{computed by } \texttt{torch.autograd}$$

In [ ]:
# ===============================================
# 4. PINN Network Definition
# ===============================================

class PINN(nn.Module):
    def __init__(self, layers=[2, 64, 64, 64, 64, 1]):
        super().__init__()
        modules = []
        for i in range(len(layers) - 2):
            modules.append(nn.Linear(layers[i], layers[i+1]))
            modules.append(nn.Tanh())
        modules.append(nn.Linear(layers[-2], layers[-1]))
        self.net = nn.Sequential(*modules)
        
        # Xavier initialization
        for m in self.net.modules():
            if isinstance(m, nn.Linear):
                nn.init.xavier_normal_(m.weight)
                nn.init.zeros_(m.bias)
    
    def forward(self, x, t):
        inputs = torch.cat([x, t], dim=1)
        return self.net(inputs)

def compute_pde_residual(model, x, t, alpha=0.1):
    x.requires_grad_(True)
    t.requires_grad_(True)
    
    u = model(x, t)
    
    # du/dt
    u_t = torch.autograd.grad(u, t, grad_outputs=torch.ones_like(u), 
                               create_graph=True)[0]
    
    # du/dx
    u_x = torch.autograd.grad(u, x, grad_outputs=torch.ones_like(u), 
                               create_graph=True)[0]
    
    # d²u/dx²
    u_xx = torch.autograd.grad(u_x, x, grad_outputs=torch.ones_like(u_x),
                                create_graph=True)[0]
    
    residual = u_t - alpha * u_xx
    return residual


model = PINN().to(device)
print(f'PINN parameters: {sum(p.numel() for p in model.parameters()):,}')
print(f'Architecture: 2 → 64 → 64 → 64 → 64 → 1')

---
## 5. Training the PINN

### Loss Function Components

1. **PDE Loss**: $\mathcal{L}_{\text{PDE}} = \frac{1}{N_f} \sum_{i=1}^{N_f} \left| r(x_f^{(i)}, t_f^{(i)}) \right|^2$
2. **IC Loss**: $\mathcal{L}_{\text{IC}} = \frac{1}{N_i} \sum_{i=1}^{N_i} \left| \hat{u}(x_i^{(i)}, 0) - \sin(\pi x_i^{(i)}) \right|^2$
3. **BC Loss**: $\mathcal{L}_{\text{BC}} = \frac{1}{N_b} \sum_{i=1}^{N_b} \left( \left| \hat{u}(0, t_b^{(i)}) \right|^2 + \left| \hat{u}(1, t_b^{(i)}) \right|^2 \right)$

$$\mathcal{L}_{\text{total}} = \lambda_1 \mathcal{L}_{\text{PDE}} + \lambda_2 \mathcal{L}_{\text{IC}} + \lambda_3 \mathcal{L}_{\text{BC}}$$

In [ ]:
# ===============================================
# 5. Training Setup
# ===============================================

# Collocation points (PDE residual points)
N_f = 10000
x_f = torch.rand(N_f, 1, device=device)
t_f = torch.rand(N_f, 1, device=device)

# Initial condition points
N_i = 200
x_i = torch.rand(N_i, 1, device=device)
t_i = torch.zeros(N_i, 1, device=device)
u_i = torch.sin(np.pi * x_i)

# Boundary condition points
N_b = 200
t_b = torch.rand(N_b, 1, device=device)
x_b0 = torch.zeros(N_b, 1, device=device)
x_b1 = torch.ones(N_b, 1, device=device)
u_b0 = torch.zeros(N_b, 1, device=device)
u_b1 = torch.zeros(N_b, 1, device=device)

# Validation grid
x_val = torch.linspace(0, 1, nx, device=device).reshape(-1, 1)
t_val = torch.linspace(0, 1, nt, device=device).reshape(-1, 1)
X_val, T_val = torch.meshgrid(x_val.squeeze(), t_val.squeeze(), indexing='ij')
U_exact_tensor = torch.sin(np.pi * X_val) * torch.exp(-alpha * np.pi**2 * T_val)

optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=2000, gamma=0.5)

n_epochs = 5000

history = {'loss': [], 'pde_loss': [], 'ic_loss': [], 'bc_loss': [], 'val_error': []}

print('Training setup complete. Starting training...')

In [ ]:
# ===============================================
# 6. Training Loop
# ===============================================

pbar = range(n_epochs)

for epoch in pbar:
    optimizer.zero_grad()
    
    # PDE loss
    residual = compute_pde_residual(model, x_f, t_f)
    loss_pde = torch.mean(residual ** 2)
    
    # IC loss
    u_pred_ic = model(x_i, t_i)
    loss_ic = torch.mean((u_pred_ic - u_i) ** 2)
    
    # BC loss
    u_pred_b0 = model(x_b0, t_b)
    u_pred_b1 = model(x_b1, t_b)
    loss_bc = torch.mean(u_pred_b0 ** 2) + torch.mean(u_pred_b1 ** 2)
    
    # Total loss
    loss = loss_pde + 10 * loss_ic + 10 * loss_bc
    
    loss.backward()
    optimizer.step()
    scheduler.step()
    
    # Logging
    history['loss'].append(loss.item())
    history['pde_loss'].append(loss_pde.item())
    history['ic_loss'].append(loss_ic.item())
    history['bc_loss'].append(loss_bc.item())
    
    if epoch % 500 == 0 or epoch == n_epochs - 1:
        with torch.no_grad():
            U_pred = model(X_val.reshape(-1, 1), T_val.reshape(-1, 1)).reshape(nx, nt)
            val_error = torch.mean((U_pred - U_exact_tensor) ** 2).item()
        history['val_error'].append(val_error)
        print(f'Epoch {epoch:5d} | Loss: {loss.item():.6e} | PDE: {loss_pde.item():.6e} | ' +
              f'IC: {loss_ic.item():.6e} | BC: {loss_bc.item():.6e} | Val MSE: {val_error:.6e}')

print('\nTraining complete!')

---
## 7. Training Convergence

In [ ]:
# ===============================================
# 7. Loss Curves
# ===============================================

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

epochs = np.arange(len(history['loss']))

axes[0].semilogy(epochs, history['loss'], 'b-', lw=1, label='Total Loss')
axes[0].semilogy(epochs, history['pde_loss'], 'r-', lw=1, alpha=0.7, label='PDE Loss')
axes[0].semilogy(epochs, history['ic_loss'], 'g-', lw=1, alpha=0.7, label='IC Loss')
axes[0].semilogy(epochs, history['bc_loss'], 'orange', lw=1, alpha=0.7, label='BC Loss')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss (log scale)')
axes[0].set_title('Training Loss Convergence')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

val_epochs = np.linspace(0, n_epochs - 1, len(history['val_error']))
axes[1].semilogy(val_epochs, history['val_error'], 'm-o', lw=2, markersize=4)
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Validation MSE (log scale)')
axes[1].set_title('Validation Error vs Analytical')
axes[1].grid(True, alpha=0.3)

plt.suptitle('Training Convergence', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print(f"Final validation MSE: {history['val_error'][-1]:.6e}")

---
## 8. PINN Prediction vs Analytical Solution

In [ ]:
# ===============================================
# 8. Predict and Compare
# ===============================================

with torch.no_grad():
    U_pred = model(X_val.reshape(-1, 1), T_val.reshape(-1, 1)).reshape(nx, nt).cpu().numpy()

U_exact_np = U_exact_tensor.cpu().numpy()
error_map = np.abs(U_pred - U_exact_np)

print(f'PINN Prediction range: [{U_pred.min():.4f}, {U_pred.max():.4f}]')
print(f'Exact range:           [{U_exact_np.min():.4f}, {U_exact_np.max():.4f}]')
print(f'Max error: {error_map.max():.4f}')
print(f'Mean error: {error_map.mean():.4f}')

In [ ]:
# ===============================================
# 8.1 3D Comparison
# ===============================================

fig = plt.figure(figsize=(18, 5))

ax1 = fig.add_subplot(131, projection='3d')
surf1 = ax1.plot_surface(X_grid, T_grid, U_pred, cmap='hot', edgecolor='none', alpha=0.9)
ax1.set_xlabel('x')
ax1.set_ylabel('t')
ax1.set_zlabel('u(x,t)')
ax1.set_title('PINN Prediction', fontweight='bold')
fig.colorbar(surf1, ax=ax1, shrink=0.6, label='Temperature')

ax2 = fig.add_subplot(132, projection='3d')
surf2 = ax2.plot_surface(X_grid, T_grid, U_exact_np, cmap='hot', edgecolor='none', alpha=0.9)
ax2.set_xlabel('x')
ax2.set_ylabel('t')
ax2.set_zlabel('u(x,t)')
ax2.set_title('Analytical Solution', fontweight='bold')
fig.colorbar(surf2, ax=ax2, shrink=0.6, label='Temperature')

ax3 = fig.add_subplot(133, projection='3d')
surf3 = ax3.plot_surface(X_grid, T_grid, error_map, cmap='Reds', edgecolor='none', alpha=0.9)
ax3.set_xlabel('x')
ax3.set_ylabel('t')
ax3.set_zlabel('|error|')
ax3.set_title('Absolute Error', fontweight='bold')
fig.colorbar(surf3, ax=ax3, shrink=0.6, label='Error')

plt.tight_layout()
plt.show()

In [ ]:
# ===============================================
# 8.2 Heatmap Comparison
# ===============================================

fig, axes = plt.subplots(3, 3, figsize=(16, 12))

data_list = [
    (U_pred, 'hot', 'PINN Prediction'),
    (U_exact_np, 'hot', 'Analytical Solution'),
    (error_map, 'Reds', 'Absolute Error'),
]

for row, (data, cmap, title) in enumerate(data_list):
    cf = axes[row, 0].contourf(X_grid, T_grid, data, levels=50, cmap=cmap)
    axes[row, 0].set_xlabel('x')
    axes[row, 0].set_ylabel('t')
    axes[row, 0].set_title(title, fontweight='bold')
    plt.colorbar(cf, ax=axes[row, 0], shrink=0.8)

# Error profile at specific times
snap_times = [0, 0.2, 0.5, 0.8, 1.0]
colors = plt.cm.viridis(np.linspace(0, 1, len(snap_times)))

for t_snap, color in zip(snap_times, colors):
    idx_t = int(t_snap / 1.0 * (nt - 1))
    axes[0, 1].plot(x_vals, U_pred[:, idx_t], color=color, lw=2, label=f't={t_snap:.1f}')
    axes[0, 2].plot(x_vals, U_exact_np[:, idx_t], color=color, lw=2, label=f't={t_snap:.1f}')
    axes[2, 1].plot(x_vals, error_map[:, idx_t], color=color, lw=2, label=f't={t_snap:.1f}')

axes[0, 1].set_title('PINN: u(x) at different times', fontweight='bold')
axes[0, 1].set_xlabel('x')
axes[0, 1].set_ylabel('u')
axes[0, 1].legend(fontsize=8)
axes[0, 1].grid(True, alpha=0.3)

axes[0, 2].set_title('Analytical: u(x) at different times', fontweight='bold')
axes[0, 2].set_xlabel('x')
axes[0, 2].set_ylabel('u')
axes[0, 2].legend(fontsize=8)
axes[0, 2].grid(True, alpha=0.3)

axes[2, 1].set_title('Error |u_pred - u_exact| at different times', fontweight='bold')
axes[2, 1].set_xlabel('x')
axes[2, 1].set_ylabel('Absolute Error')
axes[2, 1].legend(fontsize=8)
axes[2, 1].grid(True, alpha=0.3)

# Pointwise error at a fixed location x=0.5
mid_idx = nx // 2
axes[2, 2].plot(t_vals, U_pred[mid_idx, :], 'b-', lw=2, label='PINN')
axes[2, 2].plot(t_vals, U_exact_np[mid_idx, :], 'r--', lw=2, label='Analytical')
axes[2, 2].set_title('Temp Decay at x=0.5 (center)', fontweight='bold')
axes[2, 2].set_xlabel('t')
axes[2, 2].set_ylabel('u(0.5, t)')
axes[2, 2].legend()
axes[2, 2].grid(True, alpha=0.3)

# Hide unused subplots
axes[1, 1].axis('off')
axes[1, 2].axis('off')

plt.tight_layout()
plt.show()

print("PINN accurately captures both the spatial profile and temporal decay.")

In [ ]:
# ===============================================
# 8.3 Error Metrics
# ===============================================

from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

y_true_flat = U_exact_np.flatten()
y_pred_flat = U_pred.flatten()

mse_val = mean_squared_error(y_true_flat, y_pred_flat)
mae_val = mean_absolute_error(y_true_flat, y_pred_flat)
rmse_val = np.sqrt(mse_val)
r2_val = r2_score(y_true_flat, y_pred_flat)
max_err = np.max(np.abs(y_true_flat - y_pred_flat))
rel_l2 = np.linalg.norm(y_true_flat - y_pred_flat) / np.linalg.norm(y_true_flat)

print('=' * 50)
print('PINN Performance on 1D Heat Equation')
print('=' * 50)
print(f'  MSE:            {mse_val:.6e}')
print(f'  RMSE:           {rmse_val:.6e}')
print(f'  MAE:            {mae_val:.6e}')
print(f'  Max Error:      {max_err:.6e}')
print(f'  Relative L2:    {rel_l2:.6e}')
print(f'  R² Score:       {r2_val:.6f}')

In [ ]:
# ===============================================
# 8.4 Residual Distribution
# ===============================================

residuals = (y_true_flat - y_pred_flat)

fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))

axes[0].hist(residuals, bins=50, density=True, alpha=0.7, color='steelblue', edgecolor='black')
axes[0].axvline(x=0, color='red', ls='--', lw=2, label='Zero error')
axes[0].set_xlabel('Error')
axes[0].set_ylabel('Density')
axes[0].set_title(f'Error Distribution\nMean={residuals.mean():.4f}, Std={residuals.std():.4f}')
axes[0].legend()

im = axes[1].scatter(y_true_flat, y_pred_flat, c=np.abs(residuals), 
                      cmap='hot', alpha=0.5, s=2)
axes[1].plot([0, 1], [0, 1], 'r--', lw=2, label='Perfect prediction')
axes[1].set_xlabel('Exact u(x,t)')
axes[1].set_ylabel('PINN u(x,t)')
axes[1].set_title('Predicted vs Exact')
axes[1].legend(fontsize=9)
plt.colorbar(im, ax=axes[1], shrink=0.7, label='|Error|')

axes[2].plot(x_vals, U_pred[:, 0], 'b-', lw=2, label='PINN t=0')
axes[2].plot(x_vals, U_exact_np[:, 0], 'r--', lw=2, label='Exact t=0')
axes[2].plot(x_vals, U_pred[:, -1], 'b-', lw=1.5, alpha=0.7, label='PINN t=1')
axes[2].plot(x_vals, U_exact_np[:, -1], 'r--', lw=1.5, alpha=0.7, label='Exact t=1')
axes[2].set_xlabel('x')
axes[2].set_ylabel('u(x,t)')
axes[2].set_title('Initial (t=0) and Final (t=1) Profiles')
axes[2].legend(fontsize=8)
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# ===============================================
# 8.5 PDE Residual Validation
# ===============================================

# Check the PDE residual on a random set of points
N_check = 1000
x_check = torch.rand(N_check, 1, device=device)
t_check = torch.rand(N_check, 1, device=device)

with torch.enable_grad():
    residual_check = compute_pde_residual(model, x_check, t_check)
residual_np = residual_check.detach().cpu().numpy().flatten()

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))

axes[0].hist(residual_np, bins=50, density=True, alpha=0.7, color='steelblue', edgecolor='black')
axes[0].axvline(x=0, color='red', ls='--', lw=2)
axes[0].set_xlabel('PDE Residual')
axes[0].set_ylabel('Density')
axes[0].set_title(f'PDE Residual Distribution\nMean={residual_np.mean():.4e}, Std={residual_np.std():.4e}')

sc = axes[1].scatter(x_check.detach().cpu().numpy(), t_check.detach().cpu().numpy(), 
                      c=np.abs(residual_np), cmap='Reds', alpha=0.6, s=15)
axes[1].set_xlabel('x')
axes[1].set_ylabel('t')
axes[1].set_title('Spatial Distribution of PDE Residual')
plt.colorbar(sc, ax=axes[1], shrink=0.7, label='|Residual|')

plt.tight_layout()
plt.show()

print(f"RMS of PDE residual: {np.sqrt(np.mean(residual_np**2)):.6e}")
print("A well-trained PINN should have PDE residual close to 0 everywhere.")

---
## 9. How PINN Learns Over Time (Evolution Visualization)

Let's visualize how the PINN prediction evolves during training.

In [ ]:
# ===============================================
# 9. PINN Evolution During Training
# ===============================================

# Re-train with checkpoints at specific epochs (quick version)
model_evo = PINN().to(device)
optimizer_evo = torch.optim.Adam(model_evo.parameters(), lr=1e-3)

checkpoint_epochs = [0, 100, 500, 1000, 2000, 5000]
checkpoint_preds = []

x_plot = np.linspace(0, 1, 100)
t_plot = np.linspace(0, 1, 50)
Xp, Tp = np.meshgrid(x_plot, t_plot)

n_epochs_fast = 5000

for epoch in range(n_epochs_fast + 1):
    optimizer_evo.zero_grad()
    
    res = compute_pde_residual(model_evo, x_f, t_f)
    lpde = torch.mean(res ** 2)
    
    up_i = model_evo(x_i, t_i)
    lic = torch.mean((up_i - u_i) ** 2)
    
    up_b0 = model_evo(x_b0, t_b)
    up_b1 = model_evo(x_b1, t_b)
    lbc = torch.mean(up_b0 ** 2) + torch.mean(up_b1 ** 2)
    
    loss = lpde + 10 * lic + 10 * lbc
    loss.backward()
    optimizer_evo.step()
    
    if epoch in checkpoint_epochs:
        with torch.no_grad():
            xt = torch.tensor(np.stack([Xp.ravel(), Tp.ravel()], axis=1), 
                            dtype=torch.float32, device=device)
            up = model_evo(xt[:, 0:1], xt[:, 1:2]).reshape(50, 100).cpu().numpy()
        checkpoint_preds.append((epoch, up.copy()))
        print(f'Checkpoint at epoch {epoch}: saved prediction')

print('Evolution training complete!')

In [ ]:
# ===============================================
# 9.1 Visualize PINN Evolution
# ===============================================

fig, axes = plt.subplots(2, 3, figsize=(15, 9))
axes = axes.flatten()

for idx, (epoch, u_pred_ckpt) in enumerate(checkpoint_preds):
    cf = axes[idx].contourf(Xp, Tp, u_pred_ckpt, levels=50, cmap='hot')
    axes[idx].set_xlabel('x')
    axes[idx].set_ylabel('t')
    axes[idx].set_title(f'Epoch {epoch}', fontweight='bold')
    fig.colorbar(cf, ax=axes[idx], shrink=0.7)

plt.suptitle('PINN Prediction Evolution During Training', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print("The network starts random and gradually learns the heat diffusion pattern.")

In [ ]:
# ===============================================
# 9.2 Profile Evolution at t=0.5
# ===============================================

fig, ax = plt.subplots(figsize=(10, 6))

t_snap = 0.5
u_exact_05 = analytical_solution(x_plot, t_snap)

colors = plt.cm.viridis(np.linspace(0.2, 1.0, len(checkpoint_preds)))

for idx, (epoch, u_pred_ckpt) in enumerate(checkpoint_preds):
    t_idx = int(t_snap / 1.0 * 49)
    ax.plot(x_plot, u_pred_ckpt[t_idx, :], color=colors[idx], lw=2, 
            label=f'Epoch {epoch}')

ax.plot(x_plot, u_exact_05, 'k--', lw=2.5, label='Analytical (target)')
ax.set_xlabel('x')
ax.set_ylabel(f'u(x, t={t_snap})')
ax.set_title(f'PINN Prediction at t={t_snap} Over Training', fontweight='bold')
ax.legend(loc='best', fontsize=9)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("The prediction converges to the analytical solution as training progresses.")

---
## 10. Ablation: Effect of Network Size

How does the number of layers/neurons affect PINN accuracy?

In [ ]:
# ===============================================
# 10. Ablation Study
# ===============================================

architectures = [
    ('Tiny (2-16-1)', [2, 16, 1]),
    ('Small (2-32-32-1)', [2, 32, 32, 1]),
    ('Medium (2-64-64-1)', [2, 64, 64, 1]),
    ('Large (2-64-64-64-1)', [2, 64, 64, 64, 1]),
    ('Deep (2-64-64-64-64-1)', [2, 64, 64, 64, 64, 1]),
]

ablation_results = []

n_epochs_abl = 3000

for name, layers in architectures:
    m = PINN(layers).to(device)
    opt = torch.optim.Adam(m.parameters(), lr=1e-3)
    
    for ep in range(n_epochs_abl):
        opt.zero_grad()
        res = compute_pde_residual(m, x_f, t_f)
        lpde = torch.mean(res ** 2)
        up_i = m(x_i, t_i)
        lic = torch.mean((up_i - u_i) ** 2)
        up_b0 = m(x_b0, t_b)
        up_b1 = m(x_b1, t_b)
        lbc = torch.mean(up_b0 ** 2) + torch.mean(up_b1 ** 2)
        loss = lpde + 10 * lic + 10 * lbc
        loss.backward()
        opt.step()
    
    with torch.no_grad():
        U_ab = m(X_val.reshape(-1, 1), T_val.reshape(-1, 1)).reshape(nx, nt)
        mse_ab = torch.mean((U_ab - U_exact_tensor) ** 2).item()
    
    n_params = sum(p.numel() for p in m.parameters())
    ablation_results.append({'Architecture': name, 'Parameters': n_params, 'Test MSE': mse_ab})
    print(f'{name:35s} | Params: {n_params:5d} | MSE: {mse_ab:.6e}')

abl_df = pd.DataFrame(ablation_results)

In [ ]:
# ===============================================
# 10.1 Ablation Results
# ===============================================

fig, ax1 = plt.subplots(figsize=(10, 5))

x_pos = np.arange(len(abl_df))
bars = ax1.bar(x_pos, abl_df['Test MSE'], color='steelblue', edgecolor='black', width=0.6)
ax1.set_xticks(x_pos)
ax1.set_xticklabels(abl_df['Architecture'], fontsize=9)
ax1.set_ylabel('Test MSE (log scale)', fontsize=12)
ax1.set_yscale('log')
ax1.set_title('Effect of Network Size on PINN Accuracy', fontweight='bold')

for bar, val in zip(bars, abl_df['Test MSE']):
    ax1.text(bar.get_x() + bar.get_width()/2, bar.get_height() * 1.2,
             f'{val:.2e}', ha='center', va='bottom', fontsize=9, rotation=45)

ax2 = ax1.twinx()
ax2.plot(x_pos, abl_df['Parameters'], 'ro-', lw=2, markersize=8, label='Parameters')
ax2.set_ylabel('Number of Parameters', fontsize=12, color='red')
ax2.tick_params(axis='y', labelcolor='red')

plt.tight_layout()
plt.show()

print("Increasing network size improves accuracy, but with diminishing returns.")
print("The medium network (2-64-64-1) offers the best accuracy-to-cost ratio.")

In [ ]:
# ===============================================
# 11. Time Animation of PINN Solution
# ===============================================

fig, ax = plt.subplots(figsize=(8, 5))

line_pinn, = ax.plot([], [], 'b-', lw=2, label='PINN')
line_exact, = ax.plot([], [], 'r--', lw=2, label='Analytical')
time_text = ax.text(0.02, 0.95, '', transform=ax.transAxes, fontsize=14,
                    bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.8))

ax.set_xlim(0, 1)
ax.set_ylim(-0.05, 1.1)
ax.set_xlabel('x (space)')
ax.set_ylabel('u(x,t)')
ax.set_title('Heat Diffusion Over Time', fontweight='bold')
ax.legend(loc='upper right')
ax.grid(True, alpha=0.3)

def animate(frame):
    t_current = frame / (nt - 1)
    line_pinn.set_data(x_vals, U_pred[:, frame])
    line_exact.set_data(x_vals, U_exact_np[:, frame])
    time_text.set_text(f't = {t_current:.2f}')
    return line_pinn, line_exact, time_text

anim = FuncAnimation(fig, animate, frames=nt, interval=50, blit=True)
plt.close()

HTML(anim.to_jshtml())

---
## Summary

### What We Did

| Step | Description |
|------|------------|
| **Problem** | Solved the 1D heat equation $u_t = \alpha u_{xx}$ with $u(x,0)=\sin(\pi x)$, $u(0,t)=u(1,t)=0$ |
| **PINN** | Replaced traditional numerical solvers with a neural network |
| **Loss** | Enforced PDE residual + initial condition + boundary conditions |
| **Training** | Used Adam optimizer with collocation points sampled in the domain |
| **Validation** | Compared against analytical solution $u(x,t) = \sin(\pi x) e^{-\alpha \pi^2 t}$ |

### Key Takeaways

1. **PINN is mesh-free**: no grid generation needed, just sample points anywhere
2. **Physics as loss**: the PDE is embedded in the loss function, not just data fitting
3. **Automatic differentiation**: derivatives of the solution w.r.t. inputs are exact
4. **Accuracy**: with enough capacity, PINN matches the analytical solution (R² near 1.0)
5. **Network size matters**: too small → underfits; large enough → converges to true solution

### Key Formulas

$$
\boxed{\text{Heat Equation: } \frac{\partial u}{\partial t} = \alpha \frac{\partial^2 u}{\partial x^2}}
\quad\quad
\boxed{\text{PINN Loss: } \mathcal{L} = \lambda_1 \mathcal{L}_{\text{PDE}} + \lambda_2 \mathcal{L}_{\text{IC}} + \lambda_3 \mathcal{L}_{\text{BC}}}
$$

$$
\boxed{\text{Analytical: } u(x,t) = \sin(\pi x) \cdot e^{-\alpha \pi^2 t}}
\quad\quad
\boxed{\text{PDE Residual: } r(x,t) = \frac{\partial \hat{u}}{\partial t} - \alpha \frac{\partial^2 \hat{u}}{\partial x^2}}
$$